In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("PySpark Joins") \
.getOrCreate()

In [2]:
emp_data = [
    (1, "John Smith", -1, "2018", "10", "M", 3000),
    (2, "Sarah Rose", 1, "2010", "20", "F", 4000),
    (3, "Mike Williams", 1, "2010", "10", "M", 1000),
    (4, "Emma Jones", 2, "2005", "10", "F", 2000),
    (5, "David Brown", 2, "2010", "40", "M", 5500),
    (6, "Lisa Brown", 2, "2010", "50", "F", 4800),  # Dept 50 doesn't exist in dept table
    (7, "Tom Wilson", 3, "2015", "30", "M", 3500),
    (8, "Anna Davis", 1, "2020", "20", "F", 2800)
]

emp_columns = ["emp_id", "name", "superior_emp_id", "year_joined",
               "emp_dept_id", "gender", "salary"]


dept_data = [
    ("Finance", 10),
    ("Marketing", 20),
    ("Sales", 30),
    ("IT", 40),
    ("HR", 60)  # No employees in HR department
]

dept_columns = ["dept_name", "dept_id"]

empDF = spark.createDataFrame(data=emp_data, schema=emp_columns)
deptDF = spark.createDataFrame(data=dept_data, schema=dept_columns)

print("\n📊 EMPLOYEE DATASET:")
print("-" * 50)
empDF.show(truncate=False)
print(f"Total Employees: {empDF.count()}")

print("\n🏢 DEPARTMENT DATASET:")
print("-" * 50)
deptDF.show(truncate=False)
print(f"Total Departments: {deptDF.count()}")


📊 EMPLOYEE DATASET:
--------------------------------------------------
+------+-------------+---------------+-----------+-----------+------+------+
|emp_id|name         |superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+-------------+---------------+-----------+-----------+------+------+
|1     |John Smith   |-1             |2018       |10         |M     |3000  |
|2     |Sarah Rose   |1              |2010       |20         |F     |4000  |
|3     |Mike Williams|1              |2010       |10         |M     |1000  |
|4     |Emma Jones   |2              |2005       |10         |F     |2000  |
|5     |David Brown  |2              |2010       |40         |M     |5500  |
|6     |Lisa Brown   |2              |2010       |50         |F     |4800  |
|7     |Tom Wilson   |3              |2015       |30         |M     |3500  |
|8     |Anna Davis   |1              |2020       |20         |F     |2800  |
+------+-------------+---------------+-----------+-----------+------+------+

Tot

---

## Basic Syntax

```python
DataFrame.join(other, on=None, how='inner')
```

## Parameter Details

### 1. `other` (Required)
- **Type**: DataFrame
- **Description**: The right-side DataFrame to join with
- **Example**: `df1.join(df2, ...)`

### 2. `on` (Optional)
- **Type**: String, List, or Column Expression
- **Default**: None (natural join on common columns)
- **Options**:
  - **String**: `on="column_name"` - Single column join
  - **List**: `on=["col1", "col2"]` - Multiple column join
  - **Expression**: `on=df1.id == df2.user_id` - Custom join condition
  - **None**: Joins on all columns with the same name

### 3. `how` (Optional)
- **Type**: String
- **Default**: `'inner'`
- **Valid Values**: All case-insensitive

| Join Type | Aliases | Description |
|-----------|---------|-------------|
| `'inner'` | - | Returns only matching records |
| `'left'` | `'left_outer'` | All left + matching right |
| `'right'` | `'right_outer'` | All right + matching left |
| `'outer'` | `'full'`, `'full_outer'` | All records from both tables |
| `'left_semi'` | - | Left records with matches (left columns only) |
| `'left_anti'` | - | Left records without matches |
| `'cross'` | - | Cartesian product |



### Basic Join Patterns

```python
# 1. Simple inner join on common column
df1.join(df2)  # Natural join

# 2. Inner join on specific column
df1.join(df2, on="id")

# 3. Left join with specific column
df1.join(df2, on="id", how="left")

# 4. Multiple column join
df1.join(df2, on=["id", "department"])

# 5. Custom join condition
df1.join(df2, on=df1.emp_id == df2.employee_id)

# 6. Complex join conditions
df1.join(df2, on=(df1.dept_id == df2.dept_id) & (df1.salary > 5000))
```



### Advanced Syntax Patterns

```python
# Chain multiple joins
result = df1.join(df2, on="id") \
            .join(df3, on="category") \
            .join(df4, on="region", how="left")

# Join with column aliasing
from pyspark.sql.functions import col
df1.alias("emp").join(
    df2.alias("dept"),
    col("emp.dept_id") == col("dept.id")
)

# Join with broadcast hint (for small tables)
from pyspark.sql.functions import broadcast
df1.join(broadcast(df2), on="id")
```


In [5]:
empDF.show()
deptDF.show()

+------+-------------+---------------+-----------+-----------+------+------+
|emp_id|         name|superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+-------------+---------------+-----------+-----------+------+------+
|     1|   John Smith|             -1|       2018|         10|     M|  3000|
|     2|   Sarah Rose|              1|       2010|         20|     F|  4000|
|     3|Mike Williams|              1|       2010|         10|     M|  1000|
|     4|   Emma Jones|              2|       2005|         10|     F|  2000|
|     5|  David Brown|              2|       2010|         40|     M|  5500|
|     6|   Lisa Brown|              2|       2010|         50|     F|  4800|
|     7|   Tom Wilson|              3|       2015|         30|     M|  3500|
|     8|   Anna Davis|              1|       2020|         20|     F|  2800|
+------+-------------+---------------+-----------+-----------+------+------+

+---------+-------+
|dept_name|dept_id|
+---------+-------+
|  Finance|    

## 1. PySpark **Inner Join** DataFrame

The default join in PySpark is the inner join, commonly used to retrieve data from two or more DataFrames based on a shared key. An Inner join combines two DataFrames based on the key (common column) provided and results in rows where there is a matching found. Rows from both DataFrames are dropped with a non-matching key.



In [3]:
# Inner join
empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"inner") \
     .show(truncate=False)

+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name         |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|1     |John Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Mike Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Emma Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|2     |Sarah Rose   |1              |2010       |20         |F     |4000  |Marketing|20     |
|8     |Anna Davis   |1              |2020       |20         |F     |2800  |Marketing|20     |
|7     |Tom Wilson   |3              |2015       |30         |M     |3500  |Sales    |30     |
|5     |David Brown  |2              |2010       |40         |M     |5500  |IT       |40     |
+------+-------------+---------------+-----------+

## 2. PySpark **Left Outer** Join

`Left` a.k.a `Leftouter` join returns all rows from the left dataset regardless of match found on the right dataset when join expression doesn’t match, it assigns null for that record and drops records from right where match not found.

In [7]:
empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"left") \
    .show(truncate=False)


empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"leftouter") \
    .show(truncate=False)

+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name         |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|1     |John Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Mike Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Emma Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|2     |Sarah Rose   |1              |2010       |20         |F     |4000  |Marketing|20     |
|6     |Lisa Brown   |2              |2010       |50         |F     |4800  |NULL     |NULL   |
|7     |Tom Wilson   |3              |2015       |30         |M     |3500  |Sales    |30     |
|8     |Anna Davis   |1              |2020       |20         |F     |2800  |Marketing|20     |
|5     |David Brown  |2              |2010       |

## 3. Right **Outer** Join

`Right` a.k.a `Rightouter` join is opposite of `left` join, here it returns all rows from the right dataset regardless of math found on the left dataset, when join expression doesn’t match, it assigns null for that record and drops records from left where match not found.

In [8]:
empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"right") \
   .show(truncate=False)

# empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"rightouter") \
#    .show(truncate=False)

+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name         |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|4     |Emma Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|3     |Mike Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|1     |John Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|8     |Anna Davis   |1              |2020       |20         |F     |2800  |Marketing|20     |
|2     |Sarah Rose   |1              |2010       |20         |F     |4000  |Marketing|20     |
|7     |Tom Wilson   |3              |2015       |30         |M     |3500  |Sales    |30     |
|NULL  |NULL         |NULL           |NULL       |NULL       |NULL  |NULL  |HR       |60     |
|5     |David Brown  |2              |2010       |

## 4. PySpark **Full Outer** Join

`Outer` a.k.a `full`, `fullouter` join in PySpark combines the results of both left and right outer joins, ensuring that all records from both DataFrames are included in the resulting DataFrame. It includes all rows from both DataFrames and fills in missing values with nulls where there is no match. In other words, it merges the DataFrames based on a common key, but retains all rows from both DataFrames, even if there’s no match. This join type is useful when you want to preserve all the information from both datasets, regardless of whether there’s a match on the key or not.

In [9]:
empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"outer") \
    .show(truncate=False)

# empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"full") \
#     .show(truncate=False)
# empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"fullouter") \
#     .show(truncate=False)

+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name         |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+-------------+---------------+-----------+-----------+------+------+---------+-------+
|1     |John Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Mike Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Emma Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|2     |Sarah Rose   |1              |2010       |20         |F     |4000  |Marketing|20     |
|8     |Anna Davis   |1              |2020       |20         |F     |2800  |Marketing|20     |
|7     |Tom Wilson   |3              |2015       |30         |M     |3500  |Sales    |30     |
|5     |David Brown  |2              |2010       |40         |M     |5500  |IT       |40     |
|6     |Lisa Brown   |2              |2010       |

## 5. **Left Semi** Join

A Left Semi Join in PySpark returns only the rows from the left DataFrame (the first DataFrame mentioned in the join operation) where there is a match with the right DataFrame (the second DataFrame). It does not include any columns from the right DataFrame in the resulting DataFrame. This join type is useful when you only want to filter rows from the left DataFrame based on whether they have a matching key in the right DataFrame.

Left Semi Join can also be achieved by selecting only the columns from the left dataset from the result of the inner join

In [10]:
empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"leftsemi") \
   .show(truncate=False)

+------+-------------+---------------+-----------+-----------+------+------+
|emp_id|name         |superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+-------------+---------------+-----------+-----------+------+------+
|1     |John Smith   |-1             |2018       |10         |M     |3000  |
|3     |Mike Williams|1              |2010       |10         |M     |1000  |
|4     |Emma Jones   |2              |2005       |10         |F     |2000  |
|2     |Sarah Rose   |1              |2010       |20         |F     |4000  |
|8     |Anna Davis   |1              |2020       |20         |F     |2800  |
|7     |Tom Wilson   |3              |2015       |30         |M     |3500  |
|5     |David Brown  |2              |2010       |40         |M     |5500  |
+------+-------------+---------------+-----------+-----------+------+------+



## 6. **Left Anti** Join
A Left Anti Join in PySpark returns only the rows from the left DataFrame (the first DataFrame mentioned in the join operation) where there is no match with the right DataFrame (the second DataFrame). It excludes any rows from the left DataFrame that have a corresponding key in the right DataFrame. This join type is useful when you want to filter out rows from the left DataFrame that have matching keys in the right DataFrame.

In [11]:
empDF.join(deptDF,empDF.emp_dept_id ==  deptDF.dept_id,"leftanti") \
   .show(truncate=False)

+------+----------+---------------+-----------+-----------+------+------+
|emp_id|name      |superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+----------+---------------+-----------+-----------+------+------+
|6     |Lisa Brown|2              |2010       |50         |F     |4800  |
+------+----------+---------------+-----------+-----------+------+------+



## PySpark **Self Join**
Joins are not complete without a self join, Though there is no self-join type available in PySpark, we can use any of the above-explained join types to join DataFrame to itself. below example use inner self join.

In [12]:
from pyspark.sql.functions import col

empDF.alias("emp1").join(empDF.alias("emp2"), \
    col("emp1.superior_emp_id") == col("emp2.emp_id"),"inner") \
    .select(col("emp1.emp_id"),col("emp1.name"), \
      col("emp2.emp_id").alias("superior_emp_id"), \
      col("emp2.name").alias("superior_emp_name")) \
   .show(truncate=False)

+------+-------------+---------------+-----------------+
|emp_id|name         |superior_emp_id|superior_emp_name|
+------+-------------+---------------+-----------------+
|2     |Sarah Rose   |1              |John Smith       |
|3     |Mike Williams|1              |John Smith       |
|8     |Anna Davis   |1              |John Smith       |
|4     |Emma Jones   |2              |Sarah Rose       |
|5     |David Brown  |2              |Sarah Rose       |
|6     |Lisa Brown   |2              |Sarah Rose       |
|7     |Tom Wilson   |3              |Mike Williams    |
+------+-------------+---------------+-----------------+

